# AnyWidget bridge — five gates

Each gate proves one hop. Nothing is assumed; the gates are run in order
so a failure tells us exactly which layer broke.

1. Python executes in the kernel
2. `anywidget` installs hermetically from the bundled piplite index
3. The widget's JS frontend renders (Python → comm → JS)
4. `model.send()` from Python reaches the widget's `msg:custom` listener
5. The widget's JS reaches the parent page via `postMessage`

The widget must be **displayed** (gate 3) before it is pinged (gate 4):
anywidget only delivers Python→JS custom messages after the frontend has
mounted its listener.

In [ ]:
print("PYTHON_EXECUTES")

In [ ]:
%pip install anywidget ipywidgets==8.1.2
import anywidget
print("ANYWIDGET", anywidget.__version__)

In [ ]:
import anywidget
import traitlets

class BridgeProbe(anywidget.AnyWidget):
    _esm = """
    function render({ model, el }) {
      console.log("DECK_PROBE_RENDERED");
      el.textContent = "Deck probe ready";
      model.on("msg:custom", (msg) => {
        console.log("PROBE_MESSAGE", JSON.stringify(msg));
        window.parent.postMessage({
          source: "bridge-probe",
          payload: msg,
        }, "*");
      });
    }
    export default { render };
    """
    count = traitlets.Int(0).tag(sync=True)

    def ping(self, payload):
        self.send({ "kind": "ping", "payload": payload, "count": self.count })

probe = BridgeProbe(count=3)
probe

In [ ]:
# Gate 4 -- the widget was displayed in the previous cell, but its JS
# frontend mounts ASYNCHRONOUSLY after the display message. Ping too early
# and the msg:custom listener is not installed yet and the message is
# dropped. Yield once so the frontend has a chance to mount.
import asyncio
await asyncio.sleep(1)

probe.ping({ "hello": "from the kernel" })